In [7]:
import pandas as pd
import numpy as np
import torch
import torchaudio
from torchaudio import transforms
import os
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from pathlib import Path

# Load metadata
download_path = Path('data/DATA/output_chunks')
metadata_file = download_path / 'metadata.csv'
try:
    df = pd.read_csv(metadata_file)
except FileNotFoundError:
    raise FileNotFoundError(f"metadata.csv not found at {metadata_file}")
print("Columns in metadata.csv:", df.columns.tolist())
print("Sample metadata rows:", df.head().to_dict(orient='records'))

# Ensure required columns
required_columns = ['road', 'classID', 'class', 'slice_file_name']
if not all(col in df.columns for col in required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    raise KeyError(f"Missing required columns in metadata.csv: {missing}")

# Create relative_path
if 'relative_path' not in df.columns:
    df['relative_path'] = df['road'].astype(str) + '/' + df['slice_file_name'].astype(str)

# Use Path for platform-specific separators
df['relative_path'] = df['relative_path'].apply(lambda x: str(Path(x)))
df = df[['relative_path', 'classID', 'road', 'class']].copy()
df['road'] = df['road'].str.upper().str.strip()
print("Columns in df after processing:", df.columns.tolist())
print("Sample relative_path values:", df['relative_path'].head().tolist())

# Debug file paths
print("Checking download_path:", download_path)
if not download_path.exists():
    raise FileNotFoundError(f"Directory not found: {download_path}")
available_files = {str(f.relative_to(download_path)) for f in download_path.rglob("*.wav")}
print(f"Found {len(available_files)} WAV files in {download_path}")
missing_files = []
for rel_path in df['relative_path']:
    full_path = download_path / rel_path
    if not full_path.exists():
        missing_files.append(str(full_path))
if missing_files:
    print(f"Error: {len(missing_files)}/{len(df)} files missing, e.g., {missing_files[:5]}")
    print("Please verify the audio files exist in data/DATA/output_chunks/<location_name>/<slice_file_name>.")
else:
    print("All files found in metadata.csv are accessible.")

# Test loading the sample file
sample_file = download_path / "Around Arya School(1°16_32_ S 36°49_29_ E)/Around Arya School(1°16_32_ S 36°49_29_ E)--1-0.wav"
print(f"Testing sample file: {sample_file}")
try:
    sig, sr = torchaudio.load(sample_file)
    print(f"Sample file loaded successfully: {sample_file}, sample rate: {sr}, shape: {sig.shape}")
except Exception as e:
    print(f"Error loading sample file {sample_file}: {e}")

# Load Leq data
try:
    leq_data = pd.read_excel('data\FINAL DATA.xlsx', sheet_name='Noise Summary From Field')
    leq_data['Place'] = leq_data['Place'].str.upper().str.strip()
    print("Excel locations:", leq_data['Place'].unique())
except FileNotFoundError:
    raise FileNotFoundError("FINAL DATA.xlsx not found. Ensure it is in the working directory.")

# Location mapping
location_mapping = {
    'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)': 'ARYA(NGARA)',
    'THIKA ROAD(PANGANI)': 'THIKA ROAD 1',
    'THIKA ROAD 1': 'THIKA ROAD 1',
    'THIKA ROAD 2': 'THIKA ROAD 2',
    'NORTHERN BYPASS': 'NORTHERN BYPASS',
    'NORTHERN BYPASS 2': 'NORTHERN BYPASS 2',
    'KAREN C SCHOOL': 'KAREN C SCHOOL',
    # Add other mappings based on Excel locations
}
df['road'] = df['road'].map(location_mapping).fillna(df['road'])
print("Mapped road values:", df['road'].unique())

# Audio Utility Class
class AudioUtil:
    @staticmethod
    def open(audio_file):
        try:
            sig, sr = torchaudio.load(audio_file)
            return sig, sr
        except Exception as e:
            print(f"Error loading {audio_file}: {e}")
            return None

    @staticmethod
    def rechannel(aud, new_channel=1):
        if aud is None:
            return None
        sig, sr = aud
        if sig.shape[0] == new_channel:
            return aud
        if new_channel == 1:
            resig = sig[:1, :]
        else:
            resig = torch.cat([sig, sig])
        return resig, sr

    @staticmethod
    def resample(aud, newsr=22050):
        if aud is None:
            return None
        sig, sr = aud
        if sr == newsr:
            return aud
        num_channels = sig.shape[0]
        resig = torchaudio.transforms.Resample(sr, newsr)(sig[:1, :])
        if num_channels > 1:
            retwo = torchaudio.transforms.Resample(sr, newsr)(sig[1:, :])
            resig = torch.cat([resig, retwo])
        return resig, newsr

    @staticmethod
    def pad_trunc(aud, max_ms=4000):
        if aud is None:
            return None
        sig, sr = aud
        num_rows, sig_len = sig.shape
        max_len = sr // 1000 * max_ms
        if sig_len > max_len:
            sig = sig[:, :max_len]
        elif sig_len < max_len:
            pad_begin_len = (max_len - sig_len) // 2
            pad_end_len = max_len - sig_len - pad_begin_len
            pad_begin = torch.zeros((num_rows, pad_begin_len))
            pad_end = torch.zeros((num_rows, pad_end_len))
            sig = torch.cat((pad_begin, sig, pad_end), 1)
        return sig, sr

    @staticmethod
    def mfcc_feature(aud, n_mfcc=40, n_fft=1024, hop_len=512):
        if aud is None:
            return None
        try:
            sig, sr = aud
            mfcc = torchaudio.transforms.MFCC(
                sample_rate=sr,
                n_mfcc=n_mfcc,
                melkwargs={"n_fft": n_fft, "hop_length": hop_len, "n_mels": 64}
            )(sig)
            mfcc_db = torchaudio.transforms.AmplitudeToDB(top_db=80)(mfcc)
            mfcc_db = (mfcc_db - mfcc_db.mean()) / (mfcc_db.std() + 1e-6)
            return mfcc_db
        except Exception as e:
            print(f"Error computing MFCC: {e}")
            return None

    @staticmethod
    def spectral_centroid(aud, sr=22050):
        if aud is None:
            return None
        try:
            sig, sr = aud
            sig = sig.numpy().mean(axis=0)
            fft = np.abs(np.fft.fft(sig))
            freqs = np.fft.fftfreq(len(fft), 1/sr)[:len(fft)//2]
            fft = fft[:len(fft)//2]
            centroid = np.sum(freqs * fft) / (np.sum(fft) + 1e-6)
            return centroid
        except Exception as e:
            print(f"Error computing spectral centroid: {e}")
            return None

    @staticmethod
    def zero_crossing_rate(aud):
        if aud is None:
            return None
        try:
            sig, sr = aud
            sig = sig.numpy().mean(axis=0)
            zcr = np.sum(np.abs(np.diff(np.sign(sig)))) / (2 * len(sig))
            return zcr
        except Exception as e:
            print(f"Error computing ZCR: {e}")
            return None

# Dataset Class
class SoundDS(Dataset):
    def __init__(self, df, data_path, leq_data):
        self.df = df
        self.data_path = Path(data_path)
        self.leq_data = leq_data
        self.duration = 4000
        self.sr = 22050
        self.channel = 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        relative_path = self.df.loc[idx, 'relative_path']
        road = self.df.loc[idx, 'road']
        audio_file = self.data_path / relative_path
        if not audio_file.exists():
            print(f"Warning: Audio file not found: {audio_file}")
            return None
        class_id = self.df.loc[idx, 'classID']
        class_name = self.df.loc[idx, 'class']
        aud = AudioUtil.open(audio_file)
        if aud is None:
            return None
        aud = AudioUtil.resample(aud, self.sr)
        if aud is None:
            return None
        aud = AudioUtil.rechannel(aud, self.channel)
        if aud is None:
            return None
        aud = AudioUtil.pad_trunc(aud, self.duration)
        if aud is None:
            return None
        mfcc = AudioUtil.mfcc_feature(aud)
        if mfcc is None:
            return None
        centroid = AudioUtil.spectral_centroid(aud)
        if centroid is None:
            return None
        zcr = AudioUtil.zero_crossing_rate(aud)
        if zcr is None:
            return None
        leq_row = self.leq_data[self.leq_data['Place'] == road]
        leq = leq_row['Leq (dBA)'].values[0] if not leq_row.empty else 0
        return mfcc, class_id, class_name, centroid, zcr, leq, road

# Process Data and Analyze Timbre
myds = SoundDS(df, download_path, leq_data)

# Aggregate timbre features
timbre_data = []
for idx in range(len(myds)):
    result = myds[idx]
    if result is None:
        print(f"Skipping index {idx} due to processing error")
        continue
    mfcc, class_id, class_name, centroid, zcr, leq, road = result
    mfcc_mean = mfcc.mean().item() if mfcc is not None else 0
    mfcc_var = mfcc.var().item() if mfcc is not None else 0
    timbre_data.append({
        'road': road,
        'class_id': class_id,
        'class_name': class_name,
        'spectral_centroid': centroid,
        'mfcc_mean': mfcc_mean,
        'mfcc_var': mfcc_var,
        'zcr': zcr,
        'leq': leq
    })

timbre_df = pd.DataFrame(timbre_data)
print("Columns in timbre_df:", timbre_df.columns.tolist())
print(f"timbre_df rows: {len(timbre_df)}")

# Check if timbre_df is empty
if timbre_df.empty:
    raise ValueError(
        "timbre_df is empty. Check the debug output above for missing files or processing errors. "
        "Ensure audio files exist in data/DATA/output_chunks/<location_name>/<slice_file_name>."
    )

# Summarize by location and vehicle type
timbre_summary = timbre_df.groupby(['road', 'class_name']).agg({
    'spectral_centroid': 'mean',
    'mfcc_mean': 'mean',
    'mfcc_var': 'mean',
    'zcr': 'mean',
    'leq': 'mean'
}).reset_index()

# Save summary to CSV
timbre_summary.to_csv('timbre_summary.csv')

# Visualization 1: Bar Plot of Spectral Centroid by Vehicle Type
centroid_by_class = timbre_df.groupby('class_name')['spectral_centroid'].mean().reset_index()
plt.figure(figsize=(12, 6))
plt.bar(centroid_by_class['class_name'], centroid_by_class['spectral_centroid'],
        color=['#FF6384', '#36A2EB', '#FFCE56', '#4BC0C0', '#9966FF', '#FF9F40', '#C9CBCF', '#7BCF7B', '#FF5733', '#C70039', '#900C3F'])
plt.xlabel('Vehicle Type')
plt.ylabel('Mean Spectral Centroid (Hz)')
plt.title('Spectral Centroid by Vehicle Type for Tambourine Quality')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('spectral_centroid_by_vehicle.png', dpi=300)
plt.close()

# Visualization 2: Scatter Plot of Spectral Centroid vs. Leq by Location
location_summary = timbre_df.groupby('road').agg({
    'spectral_centroid': 'mean',
    'leq': 'mean'
}).reset_index()
location_summary = location_summary.sort_values('leq')

# Polynomial fit for smoothing (degree 3, similar to cubic spline)
x = np.arange(len(location_summary))
y = location_summary['spectral_centroid'].values
coeffs = np.polyfit(x, y, 3)
poly = np.poly1d(coeffs)
x_smooth = np.linspace(0, len(location_summary)-1, 100)
y_smooth = poly(x_smooth)

plt.figure(figsize=(12, 6))
plt.scatter(location_summary['leq'], location_summary['spectral_centroid'], c='blue', label='Locations', alpha=0.6)
plt.plot(location_summary['leq'].values[x_smooth.astype(int)], y_smooth, c='red', label='Smoothed Trend (Polynomial)')
plt.xlabel('Leq (dBA)')
plt.ylabel('Mean Spectral Centroid (Hz)')
plt.title('Spectral Centroid vs. Leq for Tambourine Quality Across Locations')
plt.legend()
plt.tight_layout()
plt.savefig('spectral_centroid_vs_leq.png', dpi=300)
plt.close()

# Visualization 3: Box Plot of ZCR by Vehicle Type
plt.figure(figsize=(12, 6))
zcr_by_class = [timbre_df[timbre_df['class_name'] == cls]['zcr'].values for cls in timbre_df['class_name'].unique()]
plt.boxplot(zcr_by_class, labels=timbre_df['class_name'].unique(), patch_artist=True,
            boxprops=dict(facecolor='#36A2EB', color='black'),
            medianprops=dict(color='red'))
plt.xlabel('Vehicle Type')
plt.ylabel('Zero-Crossing Rate')
plt.title('ZCR Distribution by Vehicle Type for Tambourine Quality')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('zcr_by_vehicle.png', dpi=300)
plt.close()

<>:66: SyntaxWarning: invalid escape sequence '\F'
<>:66: SyntaxWarning: invalid escape sequence '\F'
C:\Users\USER\AppData\Local\Temp\ipykernel_7244\2437358435.py:66: SyntaxWarning: invalid escape sequence '\F'
  leq_data = pd.read_excel('data\FINAL DATA.xlsx', sheet_name='Noise Summary From Field')


Columns in metadata.csv: ['slice_file_name', 'fsID', 'start', 'end', 'road', 'classID', 'class']
Sample metadata rows: [{'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-0.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 0.0, 'end': 6.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 7, 'class': 'heavy truck'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-1.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 6.0, 'end': 12.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 1, 'class': 'motorcycle'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-2.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 12.0, 'end': 18.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 1, 'class': 'motorcycle'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-3.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start

c:\Users\USER\Music\SonusAI\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
c:\Users\USER\Music\SonusAI\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decode

Excel locations: ['LANGATA LINK ROAD' 'RAILA ODINGA ROAD NEXT TO TOTAL' 'NYAYO LANGATA'
 'LIKONI ROAD' 'WINNERS CHAPEL(LIKONI ROAD)' 'LANGATA HOSPITAL' 'MMU'
 'KAREN C SCHOOL' 'JUNCTION MALL' nan 'UHURU PARK' 'JEVANJEE'
 'ARYA(NGARA)' 'OLA ENERGY WAIYAKI WAY' 'KANGEMI' 'KINOO'
 'SOUTHERN BYPASS 1' 'SOUTHERN BYPASS 2' 'NGONG ROAD' 'KAWANGWARE'
 'ICD ROAD' 'IMAARA MALL' 'KFC EMBAKASI' 'KCB UTAWALA EASTERN BYPASS'
 'MAKONGENI SHOPPING CENTRE RUAI' 'QUALITY MEAT PACKERS'
 'DAVIS & SHIRTLIFF KANGUNDO ROAD' 'BEE CENTRE' 'TOTAL ENERGIES OUTERING'
 'JOGOO ROAD' 'BBS MALL EASTLEIGH' 'THIKA ROAD(PANGANI)' 'KIAMBU ROAD'
 'RUNDA' 'RUAKA' 'KIAMBU ROAD 2' 'THOME' 'NORTHERN BYPASS'
 'OPP. KU HOSPITAL' 'NORTHERN BYPASS 2' 'THIKA ROAD 1' 'THIKA ROAD 2'
 'AROUND BABA DOGO ROAD']
Mapped road values: ['ARYA(NGARA)' 'AROUND BABA DOGO RD(1°14_51_S 36°52_26_E)'
 'AROUND JUNCTION MALL(1°17_57_S 36°45_49_E)'
 'AROUND LANGATA HOSPITAL (1°19_40_S 36°47_21_E)'
 'AROUND MMU (1°23_ 04_ 36°46_ 15_ E)'
 'BBS EASTLEIG

C:\Users\USER\AppData\Local\Temp\ipykernel_7244\2437358435.py:324: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(zcr_by_class, labels=timbre_df['class_name'].unique(), patch_artist=True,
